In [ ]:
import pandas as pd
import numpy as np

In [ ]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [ ]:
x = train_df.drop(["target", "id"],axis=1)
x_test = test_df.drop(["id"],axis=1)
y = train_df["target"]

In [ ]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = x.select_dtypes(include='object').columns

for col in categorical_cols:
    le = LabelEncoder()
    x[col] = le.fit_transform(x[col])
    x_test[col] = le.transform(x_test[col])

In [ ]:
df_corr = pd.concat([x, y], axis=1)

correlation_matrix = df_corr.corr()

correlation_with_target = correlation_matrix['target'].sort_values(ascending=False)

columns_to_drop = correlation_with_target[abs(correlation_with_target) < 0.01].index

columns_to_drop = columns_to_drop.drop('target', errors='ignore')
x = x.drop(columns=columns_to_drop)
x_test = x_test.drop(columns=columns_to_drop, errors='ignore')

print("Columns dropped:", columns_to_drop.tolist())
print("Shape of x after dropping columns:", x.shape)
print("Shape of x_test after dropping columns:", x_test.shape)

Columns dropped: ['amenities_score', 'flood_risk', 'distance_to_school_km', 'proximity_water', 'distance_nearest_park_km', 'earthquake_risk', 'altitude', 'protected_area']
Shape of x after dropping columns: (1500, 35)
Shape of x_test after dropping columns: (1200, 35)


In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42)

print("Shape of x_train:", x_train.shape)
print("Shape of x_val:", x_val.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_val:", y_val.shape)

Shape of x_train: (1200, 35)
Shape of x_val: (300, 35)
Shape of y_train: (1200,)
Shape of y_val: (300,)


In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

poly = PolynomialFeatures(degree=2, include_bias=False)
x_train_poly = poly.fit_transform(x_train)
x_val_poly = poly.transform(x_val)

print("Shape of x_train_poly:", x_train_poly.shape)
print("Shape of x_val_poly:", x_val_poly.shape)

scaler_poly = StandardScaler()
x_train_poly_scaled = scaler_poly.fit_transform(x_train_poly)
x_val_poly_scaled = scaler_poly.transform(x_val_poly)

print("Shape of x_train_poly_scaled:", x_train_poly_scaled.shape)
print("Shape of x_val_poly_scaled:", x_val_poly_scaled.shape)

Shape of x_train_poly: (1200, 665)
Shape of x_val_poly: (300, 665)
Shape of x_train_poly_scaled: (1200, 665)
Shape of x_val_poly_scaled: (300, 665)


In [ ]:
from sklearn.linear_model import ElasticNet
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

elasticnet_model_scaled = ElasticNet(alpha=0.3, l1_ratio=0.5)

elasticnet_model_scaled.fit(x_train_poly_scaled, y_train)

y_train_pred_elasticnet_scaled = elasticnet_model_scaled.predict(x_train_poly_scaled)
y_val_pred_elasticnet_scaled = elasticnet_model_scaled.predict(x_val_poly_scaled)

r2_train_elasticnet_scaled = r2_score(y_train, y_train_pred_elasticnet_scaled)
r2_val_elasticnet_scaled = r2_score(y_val, y_val_pred_elasticnet_scaled)

print(f"R2 Score on Scaled Polynomial Features Training Set (ElasticNet): {r2_train_elasticnet_scaled}")
print(f"R2 Score on Scaled Polynomial Features Validation Set (ElasticNet): {r2_val_elasticnet_scaled}")

R2 Score on Scaled Polynomial Features Training Set (ElasticNet): 0.9726065451982595
R2 Score on Scaled Polynomial Features Validation Set (ElasticNet): 0.9635926303175089


In [ ]:
x_test_poly = poly.transform(x_test)

x_test_poly_scaled = scaler_poly.transform(x_test_poly)

y_test_pred_elasticnet = elasticnet_model_scaled.predict(x_test_poly_scaled)

submission_df = pd.DataFrame({'id': test_df['id'], 'target': y_test_pred_elasticnet})

display(submission_df.head())

submission_df.to_csv('submission.csv', index=False)

print("Submission file 'submission.csv' created successfully!")

,id,target
0,id_0,8072.045409
1,id_1,18465.503418
2,id_2,13305.544970
3,id_3,28303.405931
4,id_4,17334.948101


Submission file 'submission.csv' created successfully!


In [ ]:
from sklearn.linear_model import ElasticNet
from sklearn.metrics import r2_score

elasticnet_model_poly = ElasticNet(alpha=100000.0, l1_ratio=0.9)
elasticnet_model_poly.fit(x_train_poly, y_train)

y_train_pred_poly = elasticnet_model_poly.predict(x_train_poly)
y_val_pred_poly = elasticnet_model_poly.predict(x_val_poly)

r2_train_poly = r2_score(y_train, y_train_pred_poly)
r2_val_poly = r2_score(y_val, y_val_pred_poly)

print(f"R2 Score on Polynomial Features Training Set (ElasticNet, no scaling): {r2_train_poly}")
print(f"R2 Score on Polynomial Features Validation Set (ElasticNet, no scaling): {r2_val_poly}")

R2 Score on Polynomial Features Training Set (ElasticNet, no scaling): 0.9732618981149647
R2 Score on Polynomial Features Validation Set (ElasticNet, no scaling): 0.9772382121781967


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.567e+08, tolerance: 4.116e+06
  model = cd_fast.enet_coordinate_descent(


In [ ]:
x_test_poly = poly.transform(x_test)
y_test_pred_poly = elasticnet_model_poly.predict(x_test_poly)

submission_df_poly = pd.DataFrame({'id': test_df['id'], 'target': y_test_pred_poly})

display(submission_df_poly.head())

submission_df_poly.to_csv('submission_elasticnet_poly_no_scale.csv', index=False)

print("Submission file 'submission_elasticnet_poly_no_scale.csv' created successfully!")

,id,target
0,id_0,9043.113856
1,id_1,18609.714173
2,id_2,13434.580350
3,id_3,28654.495376
4,id_4,17041.171387


Submission file 'submission_elasticnet_poly_no_scale.csv' created successfully!
